# Full DAVIS Dataset Training

In Notebook 02, I developed the initial protein–ligand binding affinity
prediction model using a smaller sample of the DAVIS dataset.

In this notebook, I extend the analysis to the full DAVIS dataset with
25,772 drug–target pairs.

I compare two modeling approaches:

1. **ESM-2 + ChemBERTa + MLP**
   - ESM-2 35M represents protein sequences
   - ChemBERTa represents drug SMILES
   - A neural network predicts binding affinity

2. **ESM-2 + Morgan Fingerprints + XGBoost**
   - The same ESM-2 protein embeddings are used
   - Morgan fingerprints represent molecular structure
   - XGBoost predicts binding affinity

The two models are evaluated on the same train, validation, and test
splits using RMSE, MAE, R², and Pearson correlation.

In [84]:
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [85]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [86]:
import pickle
import numpy as np
import pandas as pd
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [87]:
with open("/content/drive/MyDrive/drug_embeddings.pkl", "rb") as f:
    drug_embeddings = pickle.load(f)

print("Drug embeddings:", len(drug_embeddings))

Drug embeddings: 68


In [88]:
# load the complete Davis drug–target interaction dataset

!pip install -q transformers torch rdkit pandas numpy scikit-learn matplotlib PyTDC xgboost

from tdc.multi_pred import DTI

data = DTI(name="DAVIS")
data.convert_to_log(form="binding")

df = data.get_data()

print("Full dataset:", len(df), "pairs")
print("Unique drugs:", df["Drug_ID"].nunique())
print("Unique proteins:", df["Target_ID"].nunique())
print()
print(df.head())

Found local copy...
Loading...
Done!
To log space...


Full dataset: 25772 pairs
Unique drugs: 68
Unique proteins: 379

    Drug_ID                                           Drug Target_ID  \
0  11314340  Cc1[nH]nc2ccc(-c3cncc(OCC(N)Cc4ccccc4)c3)cc12      AAK1   
1  11314340  Cc1[nH]nc2ccc(-c3cncc(OCC(N)Cc4ccccc4)c3)cc12     ABL1p   
2  11314340  Cc1[nH]nc2ccc(-c3cncc(OCC(N)Cc4ccccc4)c3)cc12      ABL2   
3  11314340  Cc1[nH]nc2ccc(-c3cncc(OCC(N)Cc4ccccc4)c3)cc12     ACVR1   
4  11314340  Cc1[nH]nc2ccc(-c3cncc(OCC(N)Cc4ccccc4)c3)cc12    ACVR1B   

                                              Target         Y  
0  MKKFFDSRREQGGSGLGSGSSGGGGSTSGLGSGYIGRVFGIGRQQV...  7.365523  
1  PFWKILNPLLERGTYYYFMGQQPGKVLGDQRRPSLPALHFIKGAGK...  4.999996  
2  MVLGTVLLPPNSYGRDQDTSLCCLCTEASESALPDLTDHFASCVED...  4.999996  
3  MVDGVMILPVLIMIALPSPSMEDEKPKVNPKLYMCVCEGLSCGNED...  4.999996  
4  MAESAGASSFFPLVVLLLAGSGGSGPRGVQALLCACTSCLQANYTC...  4.999996  


In [89]:
# load ESM-2 35M as the protein sequence encoder

from transformers import AutoTokenizer, AutoModel

esm_model_name = "facebook/esm2_t12_35M_UR50D"

esm_tokenizer = AutoTokenizer.from_pretrained(esm_model_name)
esm_model = AutoModel.from_pretrained(esm_model_name).to(device)

esm_model.eval() # freeze the pretrained model

for param in esm_model.parameters():
    param.requires_grad = False

print("ESM-2 checkpoint:", esm_model_name)
print("Protein embedding dimension:", esm_model.config.hidden_size)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ESM-2 checkpoint: facebook/esm2_t12_35M_UR50D
Protein embedding dimension: 480


In [90]:
# generate ESM-2 35M embeddings for all unique proteins
# encode each protein once with the ESM-2 and reuse the embedding for all drug-target pairs

unique_proteins = (
    df[["Target_ID", "Target"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Unique proteins:", len(unique_proteins))

Unique proteins: 379


In [91]:
protein_embeddings = {}

with torch.no_grad():
    for i, row in unique_proteins.iterrows():

        target_id = row["Target_ID"]
        sequence = row["Target"]

        inputs = esm_tokenizer(
            sequence,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        outputs = esm_model(**inputs)

        # Mean pooling over amino acid tokens
        residue_embeddings = outputs.last_hidden_state[:, 1:-1, :]
        protein_embedding = residue_embeddings.mean(dim=1)

        protein_embeddings[target_id] = (
            protein_embedding.squeeze(0).cpu().numpy()
        )

        if (i + 1) % 25 == 0:
            print(f"{i + 1}/{len(unique_proteins)} proteins encoded")

print("Proteins encoded:", len(protein_embeddings))
print("Embedding size:",len(next(iter(protein_embeddings.values()))))

25/379 proteins encoded
50/379 proteins encoded
75/379 proteins encoded
100/379 proteins encoded
125/379 proteins encoded
150/379 proteins encoded
175/379 proteins encoded
200/379 proteins encoded
225/379 proteins encoded
250/379 proteins encoded
275/379 proteins encoded
300/379 proteins encoded
325/379 proteins encoded
350/379 proteins encoded
375/379 proteins encoded
Proteins encoded: 379
Embedding size: 480


In [92]:
with open(
    "/content/drive/MyDrive/protein_embeddings_esm2_35m.pkl",
    "wb"
) as f:
    pickle.dump(protein_embeddings, f)

print("Protein embeddings saved.")

Protein embeddings saved.


In [93]:
# Build the full-dataset feature matrix, for each drug-target pair, concatenate the EMS-2 protein
# embedding with the ChemBERTa drug embedding

# protein embedding + drug embedding -> X
# binding affinity -> y

X_full = []
y_full = []

for _, row in df.iterrows():
    protein_emb = protein_embeddings[row["Target_ID"]]
    drug_emb = drug_embeddings[row["Drug_ID"]]

    X_full.append(
        np.concatenate([protein_emb, drug_emb])
    )

    y_full.append(row["Y"])

X_full = np.asarray(X_full)
y_full = np.asarray(y_full)


protein_dim = len(next(iter(protein_embeddings.values())))
drug_dim = len(next(iter(drug_embeddings.values())))
fusion_dim = protein_dim + drug_dim

print("Protein embedding:", protein_dim)
print("Drug embedding:", drug_dim)
print("Combined features:", fusion_dim)

print("X shape:", X_full.shape)
print("y shape:", y_full.shape)


Protein embedding: 480
Drug embedding: 768
Combined features: 1248
X shape: (25772, 1248)
y shape: (25772,)


In [94]:
# 70% training
# 15% validation
# 15% test

from sklearn.model_selection import train_test_split

indices = np.arange(len(df))

idx_train, idx_temp = train_test_split(
    indices,
    test_size=0.30,
    random_state=42
)

idx_val, idx_test = train_test_split(
    idx_temp,
    test_size=0.50,
    random_state=42
)

X_train = X_full[idx_train]
X_val = X_full[idx_val]
X_test = X_full[idx_test]

y_train = y_full[idx_train]
y_val = y_full[idx_val]
y_test = y_full[idx_test]

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (18040, 1248) (18040,)
Validation: (3866, 1248) (3866,)
Test: (3866, 1248) (3866,)


In [95]:
# Standardize input features using training-set statistics only.
# The scaler is fitted on the training set only and then applied to the validation and test sets.

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Train mean:", X_train_scaled.mean())
print("Train std:", X_train_scaled.std())

print("\nScaled shapes:")
print("Train:", X_train_scaled.shape)
print("Validation:", X_val_scaled.shape)
print("Test:", X_test_scaled.shape)

Train mean: 5.2423893e-10
Train std: 1.0

Scaled shapes:
Train: (18040, 1248)
Validation: (3866, 1248)
Test: (3866, 1248)


In [96]:
# Training sample weights

# The DAVIS dataset contains many more low-affinity than high-affinity observations.
# use inverse-frequency weights so that less common affinity ranges have more influence during MLP training.

# The weights are calculated from the training set only and capped at 5.

bin_edges = [4.9, 5.5, 6.5, 7.5, 8.5, 10.0]

train_bins = pd.cut(
    y_train,
    bins=bin_edges,
    labels=False,
    include_lowest=True
)

bin_counts = pd.Series(train_bins).value_counts().sort_index()

bin_weights = len(y_train) / (len(bin_counts) * bin_counts)
bin_weights = bin_weights.clip(upper=5.0)

w_train = np.array(
    [bin_weights[b] for b in train_bins],
    dtype=np.float32
)

print("Training bin counts:")
print(bin_counts)

print("\nTraining bin weights:")
print(bin_weights)

print("\nWeight range:", w_train.min(), "-", w_train.max())

Training bin counts:
0    13658
1     2356
2     1248
3      561
4      217
Name: count, dtype: int64

Training bin weights:
0    0.264168
1    1.531409
2    2.891026
3    5.000000
4    5.000000
Name: count, dtype: float64

Weight range: 0.26416752 - 5.0


In [97]:
# prepare data for PyTorch
# tensor -> dataloader -> model -> loss -> optimizer -> training -> validation -> early stopping

# The training DataLoader contains the input features, pKd targets, and
# sample weights. Validation data is used without sample weighting

from torch.utils.data import TensorDataset, DataLoader

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
w_train_t = torch.tensor(w_train, dtype=torch.float32)

X_val_t = torch.tensor(X_val_scaled, dtype=torch.float32).to(device)
y_val_t = torch.tensor(y_val, dtype=torch.float32).to(device)

train_dataset = TensorDataset(
    X_train_t,
    y_train_t,
    w_train_t
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

print("Training samples:", len(train_dataset))
print("Batches per epoch:", len(train_loader))


Training samples: 18040
Batches per epoch: 141


In [98]:
# fusion MLP

import torch.nn as nn

class FusionMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


model = FusionMLP(
    input_dim=X_train_scaled.shape[1]
).to(device)

print(model)

FusionMLP(
  (net): Sequential(
    (0): Linear(in_features=1248, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=512, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.1, inplace=False)
    (6): Linear(in_features=128, out_features=1, bias=True)
  )
)


In [99]:
# loss and optimizer
# Sample weights are applied only to the training loss. Validation uses standard MSE

import torch.optim as optim

def weighted_mse(preds, targets, weights):
    return (weights * (preds - targets) ** 2).mean()


optimizer = optim.Adam(
    model.parameters(),
    lr=5e-4
)

val_criterion = nn.MSELoss()

In [100]:
# Train the model

import copy

n_epochs = 200

best_val_loss = float("inf")
best_model_state = None

for epoch in range(n_epochs):

    model.train()
    train_loss = 0.0

    for xb, yb, wb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        wb = wb.to(device)

        optimizer.zero_grad()

        preds = model(xb)
        loss = weighted_mse(preds, yb, wb)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * xb.size(0)

    train_loss /= len(train_dataset)

    # Validation
    model.eval()

    with torch.no_grad():
        val_preds = model(X_val_t)
        val_loss = val_criterion(val_preds, y_val_t).item()

    # Keep the best validation checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())

        print(
            f"Epoch {epoch+1:3d} | "
            f"Train loss: {train_loss:.4f} | "
            f"Val MSE: {val_loss:.4f}"
        )

model.load_state_dict(best_model_state)

print("\nBest validation MSE:", best_val_loss)
print("Best validation RMSE:", np.sqrt(best_val_loss))

Epoch   1 | Train loss: 2.7582 | Val MSE: 0.8292
Epoch   9 | Train loss: 0.6503 | Val MSE: 0.7190
Epoch  10 | Train loss: 0.6288 | Val MSE: 0.5754
Epoch  14 | Train loss: 0.5989 | Val MSE: 0.5310
Epoch  15 | Train loss: 0.5638 | Val MSE: 0.5144
Epoch  33 | Train loss: 0.4510 | Val MSE: 0.5108
Epoch  39 | Train loss: 0.3860 | Val MSE: 0.4611
Epoch  41 | Train loss: 0.4225 | Val MSE: 0.4405
Epoch  51 | Train loss: 0.3795 | Val MSE: 0.4223
Epoch  77 | Train loss: 0.3123 | Val MSE: 0.3975
Epoch  90 | Train loss: 0.2675 | Val MSE: 0.3886
Epoch  93 | Train loss: 0.2995 | Val MSE: 0.3811
Epoch 104 | Train loss: 0.2734 | Val MSE: 0.3736
Epoch 109 | Train loss: 0.2616 | Val MSE: 0.3694
Epoch 112 | Train loss: 0.2549 | Val MSE: 0.3638
Epoch 117 | Train loss: 0.2529 | Val MSE: 0.3503
Epoch 132 | Train loss: 0.2298 | Val MSE: 0.3436
Epoch 149 | Train loss: 0.2193 | Val MSE: 0.3403
Epoch 168 | Train loss: 0.2489 | Val MSE: 0.3337
Epoch 174 | Train loss: 0.2012 | Val MSE: 0.3337
Epoch 175 | Train lo

In [101]:
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr


# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("Train:", X_train_scaled.shape)
print("Validation:", X_val_scaled.shape)
print("Test:", X_test_scaled.shape)


# convert data to tensors
X_train_t = torch.tensor(
    X_train_scaled,
    dtype=torch.float32
)

y_train_t = torch.tensor(
    np.array(y_train),
    dtype=torch.float32
).reshape(-1)

X_val_t = torch.tensor(
    X_val_scaled,
    dtype=torch.float32
).to(device)

y_val_t = torch.tensor(
    np.array(y_val),
    dtype=torch.float32
).reshape(-1).to(device)


# model
class FusionMLP(nn.Module):

    def __init__(self, input_dim):
        super().__init__()

        self.fc1 = nn.Linear(input_dim, 512)
        self.fc2 = nn.Linear(512, 128)
        self.fc3 = nn.Linear(128, 1)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):

        x = self.relu(self.fc1(x))
        x = self.dropout(x)

        x = self.relu(self.fc2(x))
        x = self.dropout(x)

        x = self.fc3(x)

        return x.squeeze(1)


fusion_model = FusionMLP(
    X_train_scaled.shape[1]
).to(device)


# training setup
criterion = nn.MSELoss()

optimizer = optim.Adam(
    fusion_model.parameters(),
    lr=0.001
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=5
)


# data loader
train_dataset = torch.utils.data.TensorDataset(
    X_train_t,
    y_train_t
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)


# train
n_epochs = 200
patience = 30

best_val_mse = float("inf")
best_model_state = None
best_epoch = 0
patience_counter = 0


for epoch in range(1, n_epochs + 1):

    fusion_model.train()

    total_loss = 0

    for xb, yb in train_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        preds = fusion_model(xb)

        loss = criterion(preds, yb)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(xb)


    train_loss = total_loss / len(train_dataset)


    # validation
    fusion_model.eval()

    with torch.no_grad():

        val_preds = fusion_model(X_val_t)

        val_mse = criterion(
            val_preds,
            y_val_t
        ).item()


    scheduler.step(val_mse)


    # save best model
    if val_mse < best_val_mse:

        best_val_mse = val_mse
        best_epoch = epoch
        patience_counter = 0

        best_model_state = copy.deepcopy(
            fusion_model.state_dict()
        )

        print(
            f"Epoch {epoch:3d} | "
            f"Train loss: {train_loss:.4f} | "
            f"Val MSE: {val_mse:.4f}"
        )

    else:

        patience_counter += 1


    # early stopping
    if patience_counter >= patience:

        print(f"\nEarly stopping at epoch {epoch}")
        break


# restore best model
fusion_model.load_state_dict(best_model_state)
fusion_model.eval()


print()
print("Best epoch:", best_epoch)
print("Best validation MSE:", best_val_mse)
print("Best validation RMSE:", np.sqrt(best_val_mse))


# test data
X_test_t = torch.tensor(
    X_test_scaled,
    dtype=torch.float32
).to(device)

y_test_eval = np.array(
    y_test
).reshape(-1)


# test predictions
with torch.no_grad():

    test_preds = (
        fusion_model(X_test_t)
        .cpu()
        .numpy()
        .reshape(-1)
    )


# test metrics
test_mse = mean_squared_error(
    y_test_eval,
    test_preds
)

test_rmse = np.sqrt(test_mse)

test_mae = mean_absolute_error(
    y_test_eval,
    test_preds
)

test_r2 = r2_score(
    y_test_eval,
    test_preds
)

test_pearson, p_value = pearsonr(
    y_test_eval,
    test_preds
)


print()
print("Full Davis Test Results")
print("-----------------------")
print(f"MSE:       {test_mse:.4f}")
print(f"RMSE:      {test_rmse:.4f}")
print(f"MAE:       {test_mae:.4f}")
print(f"R2:        {test_r2:.4f}")
print(f"Pearson r: {test_pearson:.4f}")

Device: cuda
Train: (18040, 1248)
Validation: (3866, 1248)
Test: (3866, 1248)
Epoch   1 | Train loss: 1.7971 | Val MSE: 0.4817
Epoch   2 | Train loss: 0.7151 | Val MSE: 0.4686
Epoch   5 | Train loss: 0.6444 | Val MSE: 0.4394
Epoch   7 | Train loss: 0.5926 | Val MSE: 0.4323
Epoch   8 | Train loss: 0.5978 | Val MSE: 0.4088
Epoch   9 | Train loss: 0.5606 | Val MSE: 0.3914
Epoch  11 | Train loss: 0.5560 | Val MSE: 0.3702
Epoch  13 | Train loss: 0.5289 | Val MSE: 0.3645
Epoch  16 | Train loss: 0.5078 | Val MSE: 0.3447
Epoch  17 | Train loss: 0.4903 | Val MSE: 0.3341
Epoch  24 | Train loss: 0.4146 | Val MSE: 0.3176
Epoch  25 | Train loss: 0.4156 | Val MSE: 0.3114
Epoch  29 | Train loss: 0.4086 | Val MSE: 0.3022
Epoch  32 | Train loss: 0.3975 | Val MSE: 0.2980
Epoch  38 | Train loss: 0.3860 | Val MSE: 0.2952
Epoch  40 | Train loss: 0.3710 | Val MSE: 0.2897
Epoch  44 | Train loss: 0.3654 | Val MSE: 0.2891
Epoch  48 | Train loss: 0.3615 | Val MSE: 0.2877
Epoch  49 | Train loss: 0.3528 | Val MSE

In [102]:
# Morgan fingerprint baseline

from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np

# create Morgan fingerprints for each unique drug
unique_drugs = df[["Drug_ID", "Drug"]].drop_duplicates()

morgan_fps = {}

for _, row in unique_drugs.iterrows():
    mol = Chem.MolFromSmiles(row["Drug"])

    if mol is None:
        print("Invalid SMILES:", row["Drug_ID"])
        continue

    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=2,
        nBits=1024
    )

    morgan_fps[row["Drug_ID"]] = np.array(fp, dtype=np.float32)

print("Morgan fingerprints:", len(morgan_fps))


# combine ESM-2 protein embeddings with Morgan fingerprints
X_baseline = []

for _, row in df.iterrows():
    protein_emb = protein_embeddings[row["Target_ID"]]
    drug_fp = morgan_fps[row["Drug_ID"]]

    features = np.concatenate([protein_emb, drug_fp])
    X_baseline.append(features)

X_baseline = np.array(X_baseline, dtype=np.float32)


# check dimensions
protein_dim = len(next(iter(protein_embeddings.values())))
morgan_dim = len(next(iter(morgan_fps.values())))

print("Protein embedding:", protein_dim)
print("Morgan fingerprint:", morgan_dim)
print("Total features:", protein_dim + morgan_dim)
print("X_baseline shape:", X_baseline.shape)

Morgan fingerprints: 68


[20:25:35] DEPRECATION WARNING: please use MorganGenerator
[20:25:35] DEPRECATION WARNING: please use MorganGenerator
[20:25:35] DEPRECATION WARNING: please use MorganGenerator
[20:25:35] DEPRECATION WARNING: please use MorganGenerator
[20:25:35] DEPRECATION WARNING: please use MorganGenerator
[20:25:35] DEPRECATION WARNING: please use MorganGenerator
[20:25:35] DEPRECATION WARNING: please use MorganGenerator
[20:25:35] DEPRECATION WARNING: please use MorganGenerator
[20:25:35] DEPRECATION WARNING: please use MorganGenerator
[20:25:35] DEPRECATION WARNING: please use MorganGenerator
[20:25:35] DEPRECATION WARNING: please use MorganGenerator
[20:25:35] DEPRECATION WARNING: please use MorganGenerator
[20:25:35] DEPRECATION WARNING: please use MorganGenerator
[20:25:35] DEPRECATION WARNING: please use MorganGenerator
[20:25:35] DEPRECATION WARNING: please use MorganGenerator
[20:25:35] DEPRECATION WARNING: please use MorganGenerator
[20:25:35] DEPRECATION WARNING: please use MorganGenerat

Protein embedding: 480
Morgan fingerprint: 1024
Total features: 1504
X_baseline shape: (25772, 1504)


In [103]:
# Split baseline data

# use the same train/val/test split as the main model
X_baseline_train = X_baseline[idx_train]
X_baseline_val = X_baseline[idx_val]
X_baseline_test = X_baseline[idx_test]

y_baseline_train = y_full[idx_train]
y_baseline_val = y_full[idx_val]
y_baseline_test = y_full[idx_test]

print("Baseline train:", X_baseline_train.shape)
print("Baseline val:  ", X_baseline_val.shape)
print("Baseline test: ", X_baseline_test.shape)

Baseline train: (18040, 1504)
Baseline val:   (3866, 1504)
Baseline test:  (3866, 1504)


In [104]:
# XGBoost baseline

import xgboost as xgb

xgb_model = xgb.XGBRegressor(
    n_estimators=800,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    eval_metric="rmse",
    early_stopping_rounds=30,
    random_state=42,
    n_jobs=-1
)

# train using the validation set for early stopping
xgb_model.fit(
    X_baseline_train,
    y_baseline_train,
    eval_set=[(X_baseline_val, y_baseline_val)],
    verbose=50
)

print("\nBest iteration:", xgb_model.best_iteration)
print("Best validation RMSE:", xgb_model.best_score)

[0]	validation_0-rmse:0.78081
[50]	validation_0-rmse:0.60345
[100]	validation_0-rmse:0.57145
[150]	validation_0-rmse:0.55683
[200]	validation_0-rmse:0.54752
[250]	validation_0-rmse:0.54023
[300]	validation_0-rmse:0.53416
[350]	validation_0-rmse:0.53035
[400]	validation_0-rmse:0.52645
[450]	validation_0-rmse:0.52349
[500]	validation_0-rmse:0.52175
[550]	validation_0-rmse:0.51954
[600]	validation_0-rmse:0.51806
[650]	validation_0-rmse:0.51572
[700]	validation_0-rmse:0.51439
[750]	validation_0-rmse:0.51355
[799]	validation_0-rmse:0.51328

Best iteration: 787
Best validation RMSE: 0.513224182951779


In [105]:
# Evaluate XGBoost baseline

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr
import numpy as np

y_pred_xgb = xgb_model.predict(X_baseline_test)

mse_xgb = mean_squared_error(y_baseline_test, y_pred_xgb)
rmse_xgb = np.sqrt(mse_xgb)
mae_xgb = mean_absolute_error(y_baseline_test, y_pred_xgb)
r2_xgb = r2_score(y_baseline_test, y_pred_xgb)
pearson_xgb = pearsonr(y_baseline_test, y_pred_xgb)[0]

print("XGBoost Test Results")
print("--------------------")
print(f"MSE:       {mse_xgb:.4f}")
print(f"RMSE:      {rmse_xgb:.4f}")
print(f"MAE:       {mae_xgb:.4f}")
print(f"R2:        {r2_xgb:.4f}")
print(f"Pearson r: {pearson_xgb:.4f}")

XGBoost Test Results
--------------------
MSE:       0.2941
RMSE:      0.5423
MAE:       0.3425
R2:        0.5740
Pearson r: 0.7593


In [106]:
# Compare model results

results = pd.DataFrame({
    "Model": [
        "Fusion MLP (ESM-2 + ChemBERTa)",
        "XGBoost (ESM-2 + Morgan)"
    ],
    "MSE": [test_mse, mse_xgb],
    "RMSE": [test_rmse, rmse_xgb],
    "MAE": [test_mae, mae_xgb],
    "R2": [test_r2, r2_xgb],
    "Pearson r": [test_pearson, pearson_xgb]
})

results = results.round(4)

results

,Model,MSE,RMSE,MAE,R2,Pearson r
0,Fusion MLP (ESM-2 + ChemBERTa),0.2727,0.5222,0.3427,0.6051,0.7780
1,XGBoost (ESM-2 + Morgan),0.2941,0.5423,0.3425,0.5740,0.7593


In [107]:
# Save final model files

import os
import pickle
import shutil
import torch

# save the trained Fusion MLP
torch.save(fusion_model.state_dict(), "fusion_model_final.pt")

# save the scaler used for the ESM-2 + ChemBERTa features
with open("scaler_final.pkl", "wb") as f:
    pickle.dump(scaler, f)


# save some information about the final model
metadata = {
    "input_dim": X_train_scaled.shape[1],
    "protein_dim": 480,
    "drug_dim": 768,
    "protein_encoder": "facebook/esm2_t12_35M_UR50D",
    "molecular_encoder": "seyonec/ChemBERTa-zinc-base-v1",
    "model_architecture": "1248 -> 512 -> 128 -> 1",
    "dataset": "Davis",
    "dataset_pairs": len(df),
    "test_mse": float(test_mse),
    "test_rmse": float(test_rmse),
    "test_mae": float(test_mae),
    "test_r2": float(test_r2),
    "test_pearson": float(test_pearson),
    "split_type": "random_pair",
    "random_state": 42
}

with open("model_metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)


# copy model files to Google Drive
save_dir = "/content/drive/MyDrive/biobindai/models"
os.makedirs(save_dir, exist_ok=True)

files = [
    "fusion_model_final.pt",
    "scaler_final.pkl",
    "model_metadata.pkl"
]

for file in files:
    shutil.copy(file, os.path.join(save_dir, file))


print("Model files saved.")
print(metadata)

Model files saved.
{'input_dim': 1248, 'protein_dim': 480, 'drug_dim': 768, 'protein_encoder': 'facebook/esm2_t12_35M_UR50D', 'molecular_encoder': 'seyonec/ChemBERTa-zinc-base-v1', 'model_architecture': '1248 -> 512 -> 128 -> 1', 'dataset': 'Davis', 'dataset_pairs': 25772, 'test_mse': 0.2726538670695521, 'test_rmse': 0.5221626825708173, 'test_mae': 0.3426588131852917, 'test_r2': 0.6051268035410957, 'test_pearson': 0.7779592566813957, 'split_type': 'random_pair', 'random_state': 42}


In [108]:
# Save the XGBoost model

xgb_model.save_model(
    "/content/drive/MyDrive/biobindai/xgb_model.json"
)

print("XGBoost model saved.")


with open(
    "/content/drive/MyDrive/biobindai/morgan_fingerprints.pkl",
    "wb"
) as f:
    pickle.dump(morgan_fps, f)

print("Morgan fingerprints saved.")
print("Number of drugs:", len(morgan_fps))
print("Fingerprint dimension:", morgan_dim)

XGBoost model saved.
Morgan fingerprints saved.
Number of drugs: 68
Fingerprint dimension: 1024


In [109]:
protein_metadata = (
    df[["Target_ID", "Target"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

protein_metadata.to_csv(
    "/content/drive/MyDrive/biobindai/proteins.csv",
    index=False
)

print("Proteins saved:", len(protein_metadata))

Proteins saved: 379


In [110]:
drug_metadata = (
    df[["Drug_ID", "Drug"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

drug_metadata.to_csv(
    "/content/drive/MyDrive/biobindai/drugs.csv",
    index=False
)

print("Drugs saved:", len(drug_metadata))

Drugs saved: 68


In [111]:
# Save DAVIS data for the web application

davis_data = df[
    ["Drug_ID", "Drug", "Target_ID", "Target", "Y"]
].copy()

davis_data["split"] = ""

davis_data.loc[idx_train, "split"] = "train"
davis_data.loc[idx_val, "split"] = "validation"
davis_data.loc[idx_test, "split"] = "test"

davis_data.to_csv(
    "/content/drive/MyDrive/biobindai/davis.csv",
    index=False
)

print("DAVIS pairs saved:", len(davis_data))
print()
print(davis_data["split"].value_counts())


DAVIS pairs saved: 25772

split
train         18040
validation     3866
test           3866
Name: count, dtype: int64
